In [1]:
import os
import shutil

# 1. Define Paths
WHEEL_DIR = "/kaggle/working/wheels_fast"
# This is the path to your uploaded dataset wheel
SOURCE_WHEEL = "/kaggle/input/datasets/pjleek/onnx-runtime126-311/onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"

# 2. Reset and Create Directory
if os.path.exists(WHEEL_DIR):
    shutil.rmtree(WHEEL_DIR)
os.makedirs(WHEEL_DIR, exist_ok=True)

# 3. Move the "Tool" into the workspace
print(f"Moving ONNX Runtime 1.26.0 from Dataset to {WHEEL_DIR}...")

if os.path.exists(SOURCE_WHEEL):
    # Copy the file to your destination folder
    dest_path = os.path.join(WHEEL_DIR, os.path.basename(SOURCE_WHEEL))
    shutil.copy(SOURCE_WHEEL, dest_path)
else:
    print(f"❌ ERROR: Source wheel not found at {SOURCE_WHEEL}")
    print("Please check if the Dataset is correctly attached to the notebook.")

# 4. Final Audit
print("\n--- Final Directory Audit ---")
downloaded_files = os.listdir(WHEEL_DIR)
if any("onnxruntime" in f for f in downloaded_files):
    print(f"✅ onnxruntime 1.26.0 successfully staged: {downloaded_files[0]}")
else:
    print("❌ ERROR: onnxruntime is MISSING.")

Moving ONNX Runtime 1.26.0 from Dataset to /kaggle/working/wheels_fast...

--- Final Directory Audit ---
✅ onnxruntime 1.26.0 successfully staged: onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
%%writefile main.py
import math
import numpy as np
import os
import sys
import zipfile
import glob
from collections import defaultdict, namedtuple

# ============================================================
# 1. THE 0.1-SECOND DEPENDENCY INJECTOR (Permission Bypass)
# ============================================================
try:
    AGENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    AGENT_DIR = "/kaggle_simulations/agent"
    if not os.path.exists(AGENT_DIR): AGENT_DIR = os.getcwd()

LIB_DIR = os.path.join(AGENT_DIR, "local_lib")
os.makedirs(LIB_DIR, exist_ok=True)
sys.path.insert(0, LIB_DIR)

ort = None
try:
    import onnxruntime as ort
except ImportError:
    print("Extracting ONNXRuntime directly from wheel...", file=sys.stderr)
    wheel_paths =[
        os.path.join(AGENT_DIR, "wheels_fast"),
        "/kaggle_simulations/agent/wheels_fast",
        "/kaggle/input/datasets/pjleek/euro-step-v4/wheels_fast"
    ]
    for wp in wheel_paths:
        wheels = glob.glob(os.path.join(wp, "onnxruntime*.whl"))
        if wheels:
            wheel_file = wheels[0]
            with zipfile.ZipFile(wheel_file, 'r') as zip_ref:
                zip_ref.extractall(LIB_DIR)
            try:
                import onnxruntime as ort
                print("SUCCESS: ONNX Engine loaded instantly!", file=sys.stderr)
                break
            except Exception as e:
                pass

onnx_session = None
_logged_status = False

if ort:
    model_paths =[
        os.path.join(AGENT_DIR, "orbit_model.onnx"),
        "/kaggle_simulations/agent/orbit_model.onnx",
        "/kaggle/input/datasets/pjleek/euro-step-v4/orbit_model.onnx",
        "/kaggle/working/orbit_model.onnx"
    ]
    for mp in model_paths:
        if os.path.exists(mp):
            try:
                onnx_session = ort.InferenceSession(mp)
                break
            except: pass

# ============================================================
# 2. Engine Configuration
# ============================================================
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 0.5
HORIZON = 100
TOTAL_STEPS = 500

MIN_NN_SCORE = 0.15 

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])

# ============================================================
# 3. Core Physics Engine (Autopilot)
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

def get_target_pos(tgt, turns, ang_vel, initial_planets, comets, comet_ids):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids",[]):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): return c["paths"][idx][f_idx]
        return None
    init = initial_planets.get(tgt.id)
    if not init: return (tgt.x, tgt.y)
    r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
    if r + init.radius >= 50.0: return (tgt.x, tgt.y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, initial_planets, comets, comet_ids):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    for _ in range(5): 
        flight_dist = max(0.0, math.hypot(tx - src.x, ty - src.y) - src.radius - 0.1 - tgt.radius)
        eta = flight_dist / speed
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos: return None, 999, tx, ty
        tx, ty = pos

    angle = math.atan2(ty - src.y, tx - src.x)
    sx = src.x + math.cos(angle) * (src.radius + 0.1)
    sy = src.y + math.sin(angle) * (src.radius + 0.1)
    ex = sx + math.cos(angle) * (eta * speed)
    ey = sy + math.sin(angle) * (eta * speed)
    
    if point_to_segment_dist(CENTER_X, CENTER_Y, sx, sy, ex, ey) <= SUN_R + SUN_SAFETY: 
        return None, 999, tx, ty
        
    return angle, int(math.ceil(eta)), tx, ty

# ============================================================
# 4. Main Agent Loop (Deep Hindsight Alignment)
# ============================================================
def agent(obs, config=None):
    global _logged_status
    if not _logged_status:
        if not onnx_session: print("WARNING: ONNX Model failed to load. Falling back to Heuristic.", file=sys.stderr)
        _logged_status = True

    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    
    planets =[Planet(*p) for p in (obs.get("planets", []) if is_dict else getattr(obs, "planets",[]))]
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    initial_planets = {Planet(*p).id: Planet(*p) for p in (obs.get("initial_planets",[]) if is_dict else getattr(obs, "initial_planets",[]))}
    comets = obs.get("comets",[]) if is_dict else getattr(obs, "comets",[])
    comet_ids = set(obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids",[]))
    fleets = obs.get("fleets",[]) if is_dict else getattr(obs, "fleets",[])
    
    my_planets =[p for p in planets if p.owner == player]
    enemy_planets =[p for p in planets if p.owner not in (-1, player)]
    if not my_planets: return[]

    # --- THE V49C MACRO STATE MACHINE ---
    my_ships = sum(p.ships for p in my_planets)
    enemy_ships = sum(p.ships for p in enemy_planets)
    my_prod = sum(p.production for p in my_planets)
    enemy_prod = sum(p.production for p in enemy_planets)
    
    prod_ratio = my_prod / max(1, enemy_prod)
    
    # --- NEW CONTEXTUAL BOARD FEATURES FOR ORBITNET V2 ---
    enemy_near = enemy_ships # Total enemy presence
    best_neutral = max([p.production for p in planets if p.owner == -1] +[0])
    
    # --- V49c THREAT DETECTION ---
    threats = defaultdict(int)
    for f in fleets:
        if f[1] in (-1, player): continue
        best_d, best_tgt = float('inf'), None
        for p in planets:
            if p.id == f[5]: continue
            d = math.hypot(f[2] - p.x, f[3] - p.y)
            if d < best_d: best_d, best_tgt = d, p
        if best_tgt and best_tgt.owner == player:
            threats[best_tgt.id] += f[6]
            
    # Determine the Phase
    if my_ships > 120 and len(my_planets) < 4 and enemy_planets: state_phase = 'rush'
    elif len(my_planets) < 3: state_phase = 'expand'
    elif threats and any(t > my_ships * 0.25 for t in threats.values()): state_phase = 'counter_attack'
    elif prod_ratio > 3 and my_ships > 80: state_phase = 'crush'
    elif prod_ratio > 1.5: state_phase = 'aggressive'
    elif my_prod < enemy_prod * 0.8: state_phase = 'defend'
    else: state_phase = 'grow'

    moves =[]
    budget = {}
    evals =[]
    targeted = set()

    for src in my_planets:
        incoming_threat = threats.get(src.id, 0)
        reserve = int(incoming_threat * 1.1)
        avail = max(0, src.ships - reserve)
        
        budget[src.id] = avail
        if avail < 10: continue

        for tgt in planets:
            if tgt.owner == player: continue
            
            rough_eta = math.hypot(tgt.x - src.x, tgt.y - src.y) / fleet_speed(avail)
            base_cost = tgt.ships + (tgt.production * rough_eta if tgt.owner != -1 else 0) + 1
            
            # --- V49C SEND SIZING ---
            if state_phase in ('crush', 'rush') and tgt.owner != -1:
                cost = max(int(math.ceil(base_cost * 1.1)), int(avail * 0.75))
            elif state_phase == 'aggressive' and tgt.owner != -1:
                cost = max(int(math.ceil(base_cost * 1.1)), int(avail * 0.50))
            else:
                cost = int(math.ceil(base_cost * 1.1)) 
                
            if cost >= avail or cost < 1: continue
            
            angle, exact_eta, tx, ty = plan_flight(src, tgt, cost, ang_vel, initial_planets, comets, comet_ids)
            if angle is None or exact_eta > 45: continue 
            
            # --- 12-FEATURE GENERATION FOR ORBITNET V2 ---
            # Replaced deprecated features like is_4p and displacement to match exact train.py schema.
            feat_1 = src.production / 5.0                                    # Source value
            feat_2 = cost / max(1.0, float(src.ships))                       # Aggression ratio
            feat_3 = step / 500.0                                            # Game time
            feat_4 = enemy_near / 1000.0                                     # Defensive pressure
            feat_5 = best_neutral / 5.0                                      # Opportunity cost
            feat_6 = 1.0                                                     # Success context (Force Winner POV!)
            feat_7 = (tgt.production * max(0, 500 - step - exact_eta)) / 1000.0 # Net Present Value
            feat_8 = exact_eta / 50.0                                        # Time discount
            feat_9 = tgt.production / 5.0                                    # Target raw yield
            feat_10 = 1.0 if tgt.owner != -1 else 0.5                        # Target ownership
            feat_11 = 1.0 if (1 if src.x > 50 else -1) == (1 if tgt.x > 50 else -1) and (1 if src.y > 50 else -1) == (1 if tgt.y > 50 else -1) else 0.0 # Quadrant match
            feat_12 = fleet_speed(cost) / 6.0                                # Normalized speed
            
            if onnx_session:
                feature_vector = np.array([[
                    feat_1, feat_2, feat_3, feat_4, feat_5, feat_6, 
                    feat_7, feat_8, feat_9, feat_10, feat_11, feat_12   
                ]], dtype=np.float32)
                score = float(onnx_session.run(None, {'input': feature_vector})[0][0][0])
            else:
                # Fallback integrates v49c phases naturally
                score = 0.5 + (feat_7 / max(0.1, feat_2)) * 0.1 - (feat_8 * 0.2) + (feat_11 * 0.1)

            if score >= MIN_NN_SCORE: 
                evals.append((score, src.id, tgt.id, cost, angle))

    # Execution Loop
    evals.sort(key=lambda x: x[0], reverse=True)
    
    for score, src_id, tgt_id, cost, angle in evals:
        if budget[src_id] >= cost and tgt_id not in targeted:
            moves.append([src_id, angle, cost])
            budget[src_id] -= cost
            targeted.add(tgt_id)

    return moves

Writing main.py


In [3]:
import tarfile
import shutil
import os

working_dir = "/kaggle/working"
submission_path = os.path.join(working_dir, "submission.tar.gz")

# Hunt for the ONNX files across Kaggle's unpredictable mount paths
dataset_search_paths =[
    "/kaggle/input/datasets/pjleek/euro-step-v4",
    "/kaggle/input/euro-step-v4",
    "/kaggle/input/pjleek/euro-step-v4",
    "/kaggle/working" # In case it was generated in the same notebook session
]

def find_file(filename):
    for path in dataset_search_paths:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            return full_path
    return None

print("🔍 Locating ONNX files...")
onnx_src = find_file("orbit_model.onnx")
onnx_data_src = find_file("orbit_model.onnx.data")

if not onnx_src:
    print("❌ CRITICAL ERROR: Could not find orbit_model.onnx! Is your dataset attached to the notebook?")
else:
    # Copy to working directory
    if onnx_src != os.path.join(working_dir, "orbit_model.onnx"):
        shutil.copy(onnx_src, os.path.join(working_dir, "orbit_model.onnx"))
    print(f"✅ Found and staged: {onnx_src}")

if onnx_data_src:
    if onnx_data_src != os.path.join(working_dir, "orbit_model.onnx.data"):
        shutil.copy(onnx_data_src, os.path.join(working_dir, "orbit_model.onnx.data"))
    print(f"✅ Found and staged: {onnx_data_src}")
else:
    print("⚠️ Note: orbit_model.onnx.data not found. (If your model is small, it doesn't need this, which is fine).")

print("\n📦 Creating submission.tar.gz...")
try:
    with tarfile.open(submission_path, "w:gz") as tar:
        # 1. Add main.py
        if os.path.exists(os.path.join(working_dir, "main.py")):
            tar.add(os.path.join(working_dir, "main.py"), arcname="main.py")
            print("   ➕ Added main.py")
        else:
            print("   ❌ ERROR: main.py is missing!")
            
        # 2. Add ONNX Brain & Data
        if os.path.exists(os.path.join(working_dir, "orbit_model.onnx")):
            tar.add(os.path.join(working_dir, "orbit_model.onnx"), arcname="orbit_model.onnx")
            print("   ➕ Added orbit_model.onnx")
        else:
            print("   ❌ FAILED to add orbit_model.onnx")
            
        if os.path.exists(os.path.join(working_dir, "orbit_model.onnx.data")):
            tar.add(os.path.join(working_dir, "orbit_model.onnx.data"), arcname="orbit_model.onnx.data")
            print("   ➕ Added orbit_model.onnx.data")
            
        # 3. Add Wheels Folder
        wheels_dest = os.path.join(working_dir, "wheels_fast")
        if os.path.exists(wheels_dest):
            tar.add(wheels_dest, arcname="wheels_fast")
            print("   ➕ Added wheels_fast")
        else:
            print("   ❌ FAILED to add wheels_fast folder! Did you run the wheel downloader?")
            
    print(f"\n🎉 SUCCESS! File ready for leaderboard: {submission_path}")
        
except Exception as e:
    print(f"\n❌ FAILED to create tarball: {e}")

🔍 Locating ONNX files...
✅ Found and staged: /kaggle/input/datasets/pjleek/euro-step-v4/orbit_model.onnx
✅ Found and staged: /kaggle/input/datasets/pjleek/euro-step-v4/orbit_model.onnx.data

📦 Creating submission.tar.gz...
   ➕ Added main.py
   ➕ Added orbit_model.onnx
   ➕ Added orbit_model.onnx.data
   ➕ Added wheels_fast

🎉 SUCCESS! File ready for leaderboard: /kaggle/working/submission.tar.gz


In [4]:
import tarfile

# Define the path to your submission archive
submission_path = "/kaggle/working/submission.tar.gz"

try:
    with tarfile.open(submission_path, "r:gz") as tar:
        print(f"📦 Contents of {submission_path}:")
        for name in tar.getnames():
            print(f"  - {name}")
except Exception as e:
    print(f"❌ Error opening archive: {e}")

📦 Contents of /kaggle/working/submission.tar.gz:
  - main.py
  - orbit_model.onnx
  - orbit_model.onnx.data
  - wheels_fast
  - wheels_fast/onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
